# Twinkle 数学 GRPO 训练

本 Notebook 演示如何使用 **GRPO（Group Relative Policy Optimization）** 算法在数学问题上训练语言模型。

### 什么是 GRPO？

GRPO 是一种强化学习算法，训练流程如下：
1. 对每个数学题，让模型 **生成多个回答**
2. 用奖励函数 **评分** 每个回答（答案是否正确、格式是否规范）
3. 在同组内 **计算优势值**（advantage）：好回答得正值，差回答得负值
4. 用优势值 **更新策略**：让模型更倾向于生成好答案

### 整体流程

```
准备数据集 → 初始化训练/采样客户端 → 训练循环：
  同步权重 → 采样生成 → 计算奖励 → 计算优势 → GRPO 训练 → 日志记录
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 服务端已启动 | 需要同时配置 model 和 sampler 服务 |
| 环境变量 | 设置 `MODELSCOPE_TOKEN` |
| 依赖安装 | `pip install numpy tqdm tinker twinkle` |

## 第一步：导入依赖与全局配置

| 配置项 | 默认值 | 说明 |
|--------|--------|------|
| `BASE_MODEL` | Qwen/Qwen3.5-27B | 基座模型 |
| `NUM_GENERATIONS` | 8 | 每个 prompt 生成几条回答 |
| `MAX_NEW_TOKENS` | 4096 | 单条回答最大 token 数 |
| `LEARNING_RATE` | 1e-4 | 学习率 |
| `MAX_STEPS` | 1000 | 最大训练步数 |
| `BATCH_SIZE` | 2 | 每步的 prompt 数量（实际训练样本 = BATCH_SIZE x NUM_GENERATIONS） |
| `TEMPERATURE` | 1.0 | 采样温度，RL 训练中通常设为 1.0 保持多样性 |
| `SYNC_INTERVAL` | 1 | 每隔多少步同步权重到采样端 |
| `LORA_RANK` | 8 | LoRA 秩 |
| `DATA_NUM` | 2000 | 使用的数学题数量 |

In [ ]:
import gc
import numpy as np
import os
import re
from tinker import types
from typing import List, Tuple

from twinkle import init_tinker_client
from twinkle import get_logger
from twinkle.advantage import GRPOAdvantage
from twinkle.data_format import Message, Trajectory
from twinkle.dataloader import DataLoader
from twinkle.dataset import Dataset, DatasetMeta
from twinkle.preprocessor import Preprocessor
from twinkle.reward.base import Reward
from twinkle.metric import CompletionRewardMetric
from twinkle.template import Template

logger = get_logger()

# ========== 全局配置 ==========
BASE_MODEL = 'Qwen/Qwen3.5-27B'
NUM_GENERATIONS = 8
MAX_NEW_TOKENS = 4096
LEARNING_RATE = 1e-4
MAX_STEPS = 1000
BATCH_SIZE = 2
TEMPERATURE = 1.0
SYNC_INTERVAL = 1
LORA_RANK = 8
DATA_NUM = 2000

SYSTEM_PROMPT = ('You are a math assistant that values brevity. '
                 'Solve problems with minimal but correct reasoning.\n\n'
                 'Rules:\n'
                 '1. Use <step> </step> tags for reasoning\n'
                 '2. Final answer after ####\n\n'
                 'Example:\n<step>Key step1 -> Ket step 2 -> conclusion</step>\n#### 42')

## 第二步：定义数据预处理器

从 `competition_math` 数据集中筛选 Level 4/5 的题目（较难的题才有训练价值），提取标准答案。

预处理逻辑：
1. 只保留难度为 Level 4 或 Level 5 的题目
2. 从 solution 中用正则提取 `\boxed{...}` 里的标准答案
3. 将题目构造为 system + user 格式的对话，标准答案存入 `user_data` 供后续奖励计算

In [ ]:
class MathPreprocessor(Preprocessor):

    def __call__(self, rows):
        rows = self.map_col_to_row(rows)
        rows = [self.preprocess(row) for row in rows]
        rows = self.map_row_to_col(rows)
        return rows

    def preprocess(self, sample):
        # 只保留 Level 4/5 的题
        if sample['level'] not in ('Level 4', 'Level 5'):
            return Trajectory(messages=[], user_data=[])

        def get_boxed_answer(text):
            match = re.search(r'\\boxed{([^}]*)}', text)
            return match.group(1) if match else None

        ground_truth = get_boxed_answer(sample['solution'])
        if ground_truth is None:
            return Trajectory(messages=[], user_data=[])

        problem = sample['problem']
        return Trajectory(
            messages=[
                Message(role='system', content=SYSTEM_PROMPT),
                Message(role='user', content=problem),
            ],
            user_data=[('ground_truth', ground_truth)],
        )

## 第三步：定义奖励函数

GRPO 需要奖励函数来评判每条回答的质量。本例使用两个奖励函数：

### 准确性奖励 (MathAccuracyReward)
- 从模型输出中提取 `#### 数字` 格式的答案
- 与标准答案做数值比较
- 正确得 1.0 分，错误得 0.0 分

### 格式奖励 (MathFormatReward)
- 检查输出是否包含 `<step>...</step>` 推理标签和 `####` 答案标记
- 格式正确时，回答越短得分越高（鼓励简洁推理）
- 格式不正确得 0.0 分

**总奖励 = 准确性奖励 + 格式奖励**，最高 2.0 分。

In [ ]:
class MathAccuracyReward(Reward):
    """准确性奖励：检查模型答案是否与标准答案一致"""

    @staticmethod
    def extract_answer(completion: str) -> str:
        text = completion[-500:] if len(completion) > 500 else completion
        matches = re.findall(r'####\s*([\-\d,\.\s]+)', text)
        if matches:
            return matches[-1].replace(',', '').replace(' ', '').strip()
        return ''

    def __call__(self, trajectories: List[Trajectory], ground_truths: List[Trajectory]) -> List[float]:
        rewards = []
        for trajectory in trajectories:
            messages = trajectory.get('messages', [])
            completion = ''
            for msg in reversed(messages):
                if msg.get('role') == 'assistant':
                    completion = msg.get('content', '')
                    break

            gt = ''
            user_data = trajectory.get('user_data', [])
            if isinstance(user_data, list):
                for item in user_data:
                    if isinstance(item, (list, tuple)) and len(item) == 2:
                        if item[0] == 'ground_truth':
                            gt = str(item[1])
                            break

            predicted = self.extract_answer(completion)
            correct = False
            if predicted and gt:
                try:
                    correct = abs(float(predicted) - float(gt)) < 1e-5
                except (ValueError, OverflowError):
                    correct = predicted == gt

            rewards.append(1.0 if correct else 0.0)
        return rewards


class MathFormatReward(Reward):
    """格式奖励：检查格式并奖励简短回答"""

    def __call__(self, trajectories: List[Trajectory], ground_truths: List[Trajectory]) -> List[float]:
        rewards = []
        for trajectory in trajectories:
            messages = trajectory.get('messages', [])
            completion = ''
            for msg in reversed(messages):
                if msg.get('role') == 'assistant':
                    completion = msg.get('content', '')
                    break

            has_think = bool(re.search(r'<step>.*?</step>', completion, re.DOTALL))
            has_answer = bool(re.search(r'####\s*[\-\d,\.]+', completion))

            if not (has_think and has_answer):
                rewards.append(0.0)
            else:
                length = len(completion)
                if length <= 100:
                    rewards.append(1.0)
                else:
                    reward = max(0.0, 1.0 - (length - 100) / 2000)
                    rewards.append(reward)
        return rewards


def compute_rewards(trajectories: List[Trajectory]) -> Tuple[List[float], List[float], List[float]]:
    """计算总奖励 = 准确性 + 格式"""
    accuracy_reward_fn = MathAccuracyReward()
    format_reward_fn = MathFormatReward()
    accuracy_rewards = accuracy_reward_fn(trajectories, [])
    format_rewards = format_reward_fn(trajectories, [])
    total_rewards = [a + f for a, f in zip(accuracy_rewards, format_rewards)]
    return total_rewards, format_rewards, accuracy_rewards

## 第四步：准备数据集

加载 ModelScope 上的 `competition_math` 竞赛数学数据集，并进行预处理和编码。

- `add_generation_prompt=True`：编码时在末尾加上 assistant 前缀，准备让模型生成回答
- `truncation_strategy='delete'`：超过最大长度的样本直接删除而非截断

In [ ]:
def create_math_dataset():
    meta = DatasetMeta(
        'ms://modelscope/competition_math',
        subset_name='default',
        split='train',
        data_slice=range(DATA_NUM),
    )
    dataset = Dataset(meta)
    dataset.set_template('Template', model_id=BASE_MODEL, max_length=4096, truncation_strategy='delete')
    dataset.map(MathPreprocessor())
    dataset.filter(lambda row: bool(row['messages']))
    dataset.encode(add_generation_prompt=True)
    return dataset

dataset = create_math_dataset()
dataloader = DataLoader(dataset=dataset, batch_size=BATCH_SIZE)
template = Template(model_id=f'ms://{BASE_MODEL}')

print(f'数据集大小: {len(dataset)} 条（筛选后）')

## 第五步：初始化客户端

GRPO 训练需要两个客户端：
- **训练客户端**：负责前向反向传播和参数更新
- **采样客户端**：负责生成回答（通过 `save_weights_for_sampler` 动态创建）

采样客户端不是一开始就创建的，而是在训练过程中每隔 `SYNC_INTERVAL` 步，将最新的训练权重保存并创建一个新的采样客户端，确保采样使用的是最新的模型权重。

In [ ]:
init_tinker_client()

from tinker import ServiceClient

service_client = ServiceClient(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=os.environ.get('MODELSCOPE_TOKEN')
)

training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)

# 初始化度量和优势函数
advantage_fn = GRPOAdvantage()
metrics = CompletionRewardMetric()

sampling_params = types.SamplingParams(
    max_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=0.95,
)

print('客户端初始化完成')

## 第六步：GRPO 训练主循环

每个训练步骤包含以下阶段：

### 6.1 同步权重
将训练端最新的 LoRA 权重保存并创建采样客户端，确保采样用的是最新模型。

### 6.2 采样生成
对每个 prompt 生成 `NUM_GENERATIONS`（8）条回答。

### 6.3 计算奖励与优势
- 对每条回答计算准确性和格式奖励
- 用 `GRPOAdvantage` 在同组内标准化奖励得到优势值
- 如果同组所有回答的奖励完全相同（标准差为 0），跳过该步

### 6.4 构造训练数据
将 prompt + 生成的 completion 拼接，附上 logprobs 和 advantages，构造 GRPO 训练数据。

### 6.5 训练更新
使用 `importance_sampling` 损失函数进行前向反向传播，然后执行优化器更新。

> **注意**：训练循环较长，可根据需要调整 `MAX_STEPS`。

In [ ]:
sampling_client = None
step = 0

for batch in dataloader:
    if step >= MAX_STEPS:
        break

    metrics.reset()
    prompts = batch if isinstance(batch, list) else [batch]

    # ===== 6.1 同步权重到采样端 =====
    if step % SYNC_INTERVAL == 0:
        logger.info(f'Step {step}: Saving weights for sampler...')
        sampling_client = training_client.save_weights_and_get_sampling_client(name=f'Math-step-{step}')
        logger.info(f'Step {step}: Sampling client ready')

    if sampling_client is None:
        step += 1
        continue

    # ===== 6.2 采样生成 =====
    all_sequences = []
    all_user_data = []
    for prompt_feature in prompts:
        input_ids = prompt_feature['input_ids']
        if hasattr(input_ids, 'tolist'):
            input_ids = input_ids.tolist()
        prompt = types.ModelInput.from_ints(input_ids)
        future = sampling_client.sample(
            prompt=prompt,
            sampling_params=sampling_params,
            num_samples=NUM_GENERATIONS,
        )
        result = future.result()
        for _ in range(NUM_GENERATIONS):
            all_user_data.append(prompt_feature.get('user_data', []))
        all_sequences.extend(result.sequences)

    if not all_sequences:
        step += 1
        continue

    # ===== 6.3 构建 trajectory 并计算奖励 =====
    trajectories = []
    old_logps_list = []
    completion_lengths = []

    for idx, seq in enumerate(all_sequences):
        decoded_text = template.decode(seq.tokens, skip_special_tokens=True)
        trajectories.append({
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': 'Math problem'},
                {'role': 'assistant', 'content': decoded_text}
            ],
            'user_data': all_user_data[idx]
        })
        old_logps_list.append([lp for lp in seq.logprobs] if seq.logprobs else [])
        completion_lengths.append(len(seq.tokens))

    total_rewards, format_rewards, accuracy_rewards = compute_rewards(trajectories)
    metrics.accumulate(
        None, None,
        completion_lengths=completion_lengths,
        rewards={'total': total_rewards, 'format': format_rewards, 'accuracy': accuracy_rewards},
    )

    # ===== 6.4 计算优势值 =====
    advantages = advantage_fn(
        total_rewards,
        num_generations=NUM_GENERATIONS,
        scale='group',
    ).tolist()

    frac_zero_std = (1.0 if all(abs(a) < 1e-8 for a in advantages) else 0.0)
    if frac_zero_std == 1.0:
        logger.info(f'Step {step}: All advantages are zero, skipping training')
        step += 1
        continue

    # ===== 6.5 构造 GRPO 训练数据并训练 =====
    training_data = []
    for i, seq in enumerate(all_sequences):
        prompt_feature = prompts[i // NUM_GENERATIONS]
        prompt_ids = prompt_feature['input_ids']
        if hasattr(prompt_ids, 'tolist'):
            prompt_ids = prompt_ids.tolist()

        sampled_tokens = list(seq.tokens)
        logprobs = seq.logprobs if seq.logprobs else [0.0] * len(sampled_tokens)
        advantage = float(advantages[i])

        ob_len = len(prompt_ids) - 1
        input_tokens = prompt_ids + sampled_tokens[:-1]
        target_tokens = [0] * ob_len + sampled_tokens
        weights = [0] * ob_len + [1] * len(sampled_tokens)
        padded_advantages = [0.0] * ob_len + [advantage] * len(sampled_tokens)
        padded_logprobs = [0.0] * ob_len + logprobs

        datum = types.Datum(
            model_input=types.ModelInput.from_ints(input_tokens),
            loss_fn_inputs={
                'target_tokens': target_tokens,
                'weights': weights,
                'logprobs': types.TensorData.from_numpy(np.array(padded_logprobs, dtype=np.float32)),
                'advantages': types.TensorData.from_numpy(np.array(padded_advantages, dtype=np.float32)),
            },
        )
        training_data.append(datum)

    if not training_data:
        step += 1
        continue

    # 使用 importance_sampling 损失函数（GRPO 核心）
    fwdbwd_result = training_client.forward_backward(training_data, 'importance_sampling').result()
    optim_result = training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE)).result()

    gc.collect()

    # ===== 6.6 日志 =====
    log_dict = metrics.calculate()
    if optim_result.metrics:
        log_dict.update(optim_result.metrics)
    log_dict['train/frac_reward_zero_std'] = frac_zero_std
    log_dict['train/num_training_samples'] = len(training_data)
    logger.info(f'Step {step}: {log_dict}')
    step += 1

## 第七步：保存最终检查点

训练完成后保存最终的 LoRA 检查点，可用于后续推理（参见 `sample.ipynb`）。

In [ ]:
save_future = training_client.save_state('Math-grpo-final')
save_result = save_future.result()
logger.info(f'Saved final checkpoint to {save_result.path}')
print(f'训练完成！检查点保存在: {save_result.path}')

## 关键指标解读

训练过程中会输出以下指标：

| 指标 | 含义 | 期望趋势 |
|------|------|----------|
| `accuracy` | 回答正确率 | 逐步上升 |
| `format` | 格式正确率 | 快速达到高值 |
| `total` | 总奖励（准确性+格式） | 逐步上升 |
| `frac_reward_zero_std` | 同组奖励标准差为零的比例 | 逐步下降（说明模型在区分好坏回答） |
| `completion_lengths` | 回答平均长度 | 逐步缩短（简洁奖励的效果） |

## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| 准确率不提升 | 学习率太低/太高 | 尝试 5e-5 或 2e-4 |
| 所有 advantage 为 0 | 同组回答奖励完全相同 | 增大 NUM_GENERATIONS 或提高 temperature |
| OOM 内存不足 | 生成太长 | 减小 MAX_NEW_TOKENS 或 BATCH_SIZE |
| 采样超时 | 服务端 sampler 未配置 | 检查 server_config.yaml 中 sampler 配置 |